# NeuroScan - Brain Tumor MRI Classification
Trains EfficientNetB3, VGG16, and InceptionV3 via two-phase transfer learning.

**Dataset:** masoudnickparvar/brain-tumor-mri-dataset
**Output:** Three .pt weight files under /kaggle/working/neuroscan/

In [ ]:
# Cell 1 - GPU info
# P100 (sm_60) is incompatible with PyTorch 2.1+ on Python 3.12.
# No installable PyTorch supports both Python 3.12 and sm_60.
# train.py detects this and falls back to CPU automatically.
import subprocess, torch
r = subprocess.run(['nvidia-smi', '--query-gpu=name,compute_cap', '--format=csv,noheader'],
                   capture_output=True, text=True)
print('GPU info   :', r.stdout.strip() or 'none')
print('PyTorch    :', torch.__version__)
print('CUDA avail :', torch.cuda.is_available())
if torch.cuda.is_available():
    major, _ = torch.cuda.get_device_capability(0)
    if major < 7:
        print('NOTE: P100 sm_60 is not supported by this PyTorch. Training will use CPU.')
    else:
        print('GPU is compatible - training will use CUDA.')

In [ ]:
# Cell 2 - Find the dataset wherever Kaggle mounted it
import os, subprocess

def find_dataset_root():
    base = '/kaggle/input'
    if not os.path.exists(base):
        return None
    for item in os.listdir(base):
        path = os.path.join(base, item)
        if os.path.isdir(os.path.join(path, 'Training')):
            return path
        for sub in os.listdir(path):
            subpath = os.path.join(path, sub)
            if os.path.isdir(os.path.join(subpath, 'Training')):
                return subpath
    return None

print('Contents of /kaggle/input:')
for item in sorted(os.listdir('/kaggle/input')):
    print(' ', item)

DATA_ROOT = find_dataset_root()

if DATA_ROOT is None:
    print('\nDataset not mounted - downloading via Kaggle API...')
    os.makedirs('/kaggle/working/data', exist_ok=True)
    subprocess.run([
        'kaggle', 'datasets', 'download',
        'masoudnickparvar/brain-tumor-mri-dataset',
        '-p', '/kaggle/working/data', '--unzip'
    ], check=True)
    DATA_ROOT = find_dataset_root() or '/kaggle/working/data'

print(f'\nDataset root : {DATA_ROOT}')
print(f'Training ok  : {os.path.isdir(DATA_ROOT + "/Training")}')
print(f'Testing  ok  : {os.path.isdir(DATA_ROOT + "/Testing")}')

with open('/kaggle/working/data_root.txt', 'w') as f:
    f.write(DATA_ROOT)

In [ ]:
%%writefile /kaggle/working/train.py
"""
NeuroScan - Multi-Architecture Brain Tumor MRI Classification
EfficientNetB3, VGG16, InceptionV3 - two-phase transfer learning.
Auto-detects GPU; falls back to CPU if GPU is incompatible (e.g. P100 + PyTorch 2.1+).
"""

import os, copy, time, argparse
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, classification_report,
)
from tqdm import tqdm


def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument('--model',          type=str, default='efficientnet_b3',
                   choices=['efficientnet_b3', 'vgg16', 'inception_v3'])
    p.add_argument('--data-dir',       type=str, default='/kaggle/input/brain-tumor-mri-dataset')
    p.add_argument('--output-dir',     type=str, default='/kaggle/working/neuroscan')
    p.add_argument('--epochs-phase1',  type=int, default=10)
    p.add_argument('--epochs-phase2',  type=int, default=10)
    p.add_argument('--batch-size',     type=int, default=32)
    p.add_argument('--lr-phase1',      type=float, default=1e-3)
    p.add_argument('--lr-phase2',      type=float, default=1e-5)
    p.add_argument('--seed',           type=int, default=42)
    p.add_argument('--num-workers',    type=int, default=2)
    p.add_argument('--no-wandb',       action='store_true')
    p.add_argument('--wandb-project',  type=str, default='neuroscan')
    p.add_argument('--wandb-run-name', type=str, default=None)
    return p.parse_args()


CLASS_NAMES   = ['glioma', 'meningioma', 'notumor', 'pituitary']
NUM_CLASSES   = 4
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]


def find_data_root(data_dir):
    if os.path.isdir(os.path.join(data_dir, 'Training')):
        return data_dir
    for sub in os.listdir(data_dir):
        path = os.path.join(data_dir, sub)
        if os.path.isdir(os.path.join(path, 'Training')):
            return path
    raise FileNotFoundError(f'Could not find Training/ under {data_dir}')


def build_dataframe(root):
    records = []
    for cls in os.listdir(root):
        cls_dir = os.path.join(root, cls)
        if not os.path.isdir(cls_dir):
            continue
        for fname in os.listdir(cls_dir):
            if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                records.append({'img_path': os.path.join(cls_dir, fname), 'label': cls})
    return pd.DataFrame(records)


class BrainTumorDataset(Dataset):
    def __init__(self, df, class_names, transform=None):
        self.df        = df.reset_index(drop=True)
        self.label_map = {c: i for i, c in enumerate(class_names)}
        self.transform = transform

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        image = Image.open(row['img_path']).convert('RGB')
        label = self.label_map[row['label']]
        if self.transform: image = self.transform(image)
        return image, label


def build_loaders(data_dir, batch_size, num_workers, img_size):
    data_dir  = find_data_root(data_dir)
    train_dir = os.path.join(data_dir, 'Training')
    test_dir  = os.path.join(data_dir, 'Testing')

    train_df = build_dataframe(train_dir)
    test_df  = build_dataframe(test_dir)
    train_df = train_df[train_df['label'].isin(CLASS_NAMES)].reset_index(drop=True)
    test_df  = test_df[test_df['label'].isin(CLASS_NAMES)].reset_index(drop=True)
    print(f'Train: {len(train_df)}  Test: {len(test_df)}')

    aug = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
    basic = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
    train_loader = DataLoader(BrainTumorDataset(train_df, CLASS_NAMES, aug),
                              batch_size=batch_size, shuffle=True,
                              num_workers=num_workers, pin_memory=True)
    test_loader  = DataLoader(BrainTumorDataset(test_df, CLASS_NAMES, basic),
                              batch_size=batch_size, shuffle=False,
                              num_workers=num_workers, pin_memory=True)
    return train_loader, test_loader


def build_efficientnet_b3(num_classes, freeze_backbone):
    model = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.IMAGENET1K_V1)
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(model.classifier[1].in_features, num_classes),
    )
    if freeze_backbone:
        for param in model.features.parameters(): param.requires_grad = False
    return model

def unfreeze_efficientnet_b3(model):
    for param in model.parameters(): param.requires_grad = False
    for name, param in model.named_parameters():
        if 'features.6' in name or 'features.7' in name or 'classifier' in name:
            param.requires_grad = True


def build_vgg16(num_classes, freeze_backbone):
    model = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
    model.classifier = nn.Sequential(
        *list(model.classifier.children())[:-1],
        nn.Dropout(p=0.4),
        nn.Linear(4096, num_classes),
    )
    if freeze_backbone:
        for param in model.features.parameters(): param.requires_grad = False
    return model

def unfreeze_vgg16(model):
    for param in model.parameters(): param.requires_grad = False
    for name, param in model.named_parameters():
        if 'classifier' in name:
            param.requires_grad = True
        elif name.startswith('features.') and int(name.split('.')[1]) >= 24:
            param.requires_grad = True


def build_inception_v3(num_classes, freeze_backbone):
    model = models.inception_v3(
        weights=models.Inception_V3_Weights.IMAGENET1K_V1, aux_logits=True)
    model.AuxLogits.fc = nn.Linear(model.AuxLogits.fc.in_features, num_classes)
    model.fc = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(model.fc.in_features, num_classes),
    )
    if freeze_backbone:
        for param in model.parameters(): param.requires_grad = False
        for param in model.fc.parameters():           param.requires_grad = True
        for param in model.AuxLogits.fc.parameters(): param.requires_grad = True
    return model

def unfreeze_inception_v3(model):
    for param in model.parameters(): param.requires_grad = False
    for name, param in model.named_parameters():
        if 'Mixed_7' in name or name.startswith('fc') or 'AuxLogits' in name:
            param.requires_grad = True


def count_trainable(model):
    t = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n = sum(p.numel() for p in model.parameters())
    print(f'  Trainable: {t:,} / {n:,} ({100*t/n:.1f}%)')


def train_one_epoch(model, loader, criterion, optimizer, device, use_amp, is_inception=False):
    model.train()
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    loss_sum, correct, total = 0.0, 0, 0
    for images, labels in tqdm(loader, leave=False, desc='  train'):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda', enabled=use_amp):
            if is_inception:
                out, aux = model(images)
                loss = criterion(out, labels) + 0.4 * criterion(aux, labels)
            else:
                out  = model(images)
                loss = criterion(out, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        loss_sum += loss.item() * images.size(0)
        correct  += (out.argmax(1) == labels).sum().item()
        total    += labels.size(0)
    return loss_sum / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    for images, labels in tqdm(loader, leave=False, desc='  eval'):
        images, labels = images.to(device), labels.to(device)
        out  = model(images)
        loss = criterion(out, labels)
        loss_sum += loss.item() * images.size(0)
        correct  += (out.argmax(1) == labels).sum().item()
        total    += labels.size(0)
    return loss_sum / total, correct / total


def run_phase(model, train_loader, test_loader, criterion, optimizer,
              epochs, device, phase_name, output_dir, use_amp, wandb_run, is_inception=False):
    best_acc, best_state = 0.0, copy.deepcopy(model.state_dict())
    ckpt_dir = os.path.join(output_dir, 'checkpoints')
    os.makedirs(ckpt_dir, exist_ok=True)
    for epoch in range(1, epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, device, use_amp, is_inception)
        va_loss, va_acc = evaluate(model, test_loader, criterion, device)
        print(f'  [{phase_name}] Epoch {epoch:02d}/{epochs}  '
              f'train_loss={tr_loss:.4f}  train_acc={tr_acc:.4f}  '
              f'val_loss={va_loss:.4f}  val_acc={va_acc:.4f}  '
              f'({time.time()-t0:.1f}s)')
        if wandb_run:
            wandb_run.log({f'{phase_name}/train_loss': tr_loss, f'{phase_name}/train_acc': tr_acc,
                           f'{phase_name}/val_loss': va_loss, f'{phase_name}/val_acc': va_acc, 'epoch': epoch})
        if va_acc > best_acc:
            best_acc, best_state = va_acc, copy.deepcopy(model.state_dict())
            torch.save(best_state, os.path.join(ckpt_dir, f'{phase_name}_best.pt'))
        if epoch % 5 == 0:
            torch.save(model.state_dict(), os.path.join(ckpt_dir, f'{phase_name}_epoch{epoch:02d}.pt'))
    model.load_state_dict(best_state)
    print(f'\n  Best val acc ({phase_name}): {best_acc:.4f}\n')


@torch.no_grad()
def full_evaluation(model, loader, device):
    model.eval()
    preds, labels = [], []
    for images, lbls in loader:
        preds.extend(model(images.to(device)).argmax(1).cpu().numpy())
        labels.extend(lbls.numpy())
    y_true, y_pred = np.array(labels), np.array(preds)
    return {
        'accuracy':  round(float(accuracy_score(y_true, y_pred)), 4),
        'f1':        round(float(f1_score(y_true, y_pred, average='weighted')), 4),
        'precision': round(float(precision_score(y_true, y_pred, average='weighted', zero_division=0)), 4),
        'recall':    round(float(recall_score(y_true, y_pred, average='weighted')), 4),
        'report':    classification_report(y_true, y_pred, target_names=CLASS_NAMES),
    }


MODEL_REGISTRY = {
    'efficientnet_b3': {'build': build_efficientnet_b3, 'unfreeze': unfreeze_efficientnet_b3,
                        'img_size': 224, 'output': 'efficientnet_b3_neuroscan.pt'},
    'vgg16':           {'build': build_vgg16,           'unfreeze': unfreeze_vgg16,
                        'img_size': 224, 'output': 'vgg16_neuroscan.pt'},
    'inception_v3':    {'build': build_inception_v3,    'unfreeze': unfreeze_inception_v3,
                        'img_size': 299, 'output': 'inception_v3_neuroscan.pt'},
}


def main():
    args = parse_args()
    torch.manual_seed(args.seed)
    np.random.seed(args.seed)

    device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    use_amp = device.type == 'cuda'

    # Verify the GPU actually works before committing to CUDA mode.
    # P100 (sm_60) is incompatible with PyTorch 2.1+ on Python 3.12 - falls back to CPU.
    if device.type == 'cuda':
        try:
            torch.zeros(1, device=device)
        except Exception as e:
            print(f'GPU not usable ({type(e).__name__}): falling back to CPU.')
            device  = torch.device('cpu')
            use_amp = False

    is_inception = args.model == 'inception_v3'
    cfg          = MODEL_REGISTRY[args.model]

    print(f'\n  Model  : {args.model}')
    print(f'  Device : {device}')
    if device.type == 'cuda':
        print(f'  GPU    : {torch.cuda.get_device_name(0)}')
    print(f'  AMP    : {use_amp}\n')

    wandb_run = None
    if not args.no_wandb:
        try:
            import wandb
            wandb_run = wandb.init(project=args.wandb_project,
                                   name=args.wandb_run_name or f'{args.model}',
                                   config=vars(args))
            print(f'  W&B: {wandb_run.url}\n')
        except Exception as e:
            print(f'  WARNING: W&B unavailable ({e}). Skipping.')

    log_path = os.path.join(args.output_dir, 'logs', 'training.log')
    os.makedirs(os.path.dirname(log_path), exist_ok=True)

    print('-- Loading dataset --')
    train_loader, test_loader = build_loaders(
        args.data_dir, args.batch_size, args.num_workers, cfg['img_size'])

    print(f'\n-- Building {args.model} --')
    model     = cfg['build'](NUM_CLASSES, freeze_backbone=True).to(device)
    criterion = nn.CrossEntropyLoss()

    if args.epochs_phase1 > 0:
        print('\n-- Phase 1: Feature Extraction --')
        count_trainable(model)
        opt1 = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=args.lr_phase1)
        run_phase(model, train_loader, test_loader, criterion, opt1,
                  args.epochs_phase1, device, 'phase1', args.output_dir, use_amp, wandb_run, is_inception)

    if args.epochs_phase2 > 0:
        print('\n-- Phase 2: Fine-Tuning --')
        cfg['unfreeze'](model)
        count_trainable(model)
        opt2 = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=args.lr_phase2)
        run_phase(model, train_loader, test_loader, criterion, opt2,
                  args.epochs_phase2, device, 'phase2', args.output_dir, use_amp, wandb_run, is_inception)

    print('\n-- Final Evaluation --')
    m = full_evaluation(model, test_loader, device)
    for k in ('accuracy', 'f1', 'precision', 'recall'):
        print(f'  {k.capitalize():<12}: {m[k]}')
    print(f'\n{m["report"]}')

    if wandb_run:
        wandb_run.summary.update({f'final/{k}': m[k] for k in ('accuracy', 'f1', 'precision', 'recall')})

    out_path = os.path.join(args.output_dir, 'outputs', cfg['output'])
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    torch.save(model.state_dict(), out_path)
    print(f'\n  Weights saved: {out_path}')

    with open(log_path, 'a') as f:
        f.write(f"\nModel: {args.model}  Completed: {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Accuracy: {m['accuracy']}  F1: {m['f1']}\n\n{m['report']}\n")

    if wandb_run: wandb_run.finish()
    print('\n  Done.\n')


if __name__ == '__main__':
    main()

## Model 1 - EfficientNetB3

In [ ]:
DATA_DIR = open('/kaggle/working/data_root.txt').read().strip()
!python /kaggle/working/train.py \
    --model         efficientnet_b3 \
    --data-dir      {DATA_DIR} \
    --output-dir    /kaggle/working/neuroscan/efficientnet_b3 \
    --epochs-phase1 10 \
    --epochs-phase2 10 \
    --batch-size    32 \
    --num-workers   2 \
    --no-wandb

## Model 2 - VGG16

In [ ]:
DATA_DIR = open('/kaggle/working/data_root.txt').read().strip()
!python /kaggle/working/train.py \
    --model         vgg16 \
    --data-dir      {DATA_DIR} \
    --output-dir    /kaggle/working/neuroscan/vgg16 \
    --epochs-phase1 10 \
    --epochs-phase2 10 \
    --batch-size    32 \
    --num-workers   2 \
    --no-wandb

## Model 3 - InceptionV3

In [ ]:
DATA_DIR = open('/kaggle/working/data_root.txt').read().strip()
!python /kaggle/working/train.py \
    --model         inception_v3 \
    --data-dir      {DATA_DIR} \
    --output-dir    /kaggle/working/neuroscan/inception_v3 \
    --epochs-phase1 10 \
    --epochs-phase2 10 \
    --batch-size    32 \
    --num-workers   2 \
    --no-wandb

## Output Summary

In [ ]:
import os
print('Saved weight files:')
for root, dirs, files in os.walk('/kaggle/working/neuroscan'):
    for f in files:
        path = os.path.join(root, f)
        mb = os.path.getsize(path) / 1e6
        print(f'  {path}  ({mb:.1f} MB)')